In [2]:
from backtesting import Backtest, Strategy
import pandas as pd

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [3]:
df_forex = pd.read_csv('/home/pulpo/Documents/Trading_ML/EURUSD/EURUSD_2025.csv', 
                       sep=';', 
                       names=['DateTime', 'Open', 'High', 'Low', 'Close', 'Volume'], 
                       index_col='DateTime', 
                       parse_dates=True)

# 2. Ajuste para Forex: Backtesting.py necesita que el índice sea DatetimeIndex
df_forex.index = pd.to_datetime(df_forex.index, format='%Y%m%d %H%M%S')

# 3. OPCIONAL: Si el archivo es MUY grande, puedes recortar un mes para probar rápido
# df_forex = df_forex.loc['2025-01-01':'2025-01-31']

# limpieza aunque nose si es necesaria
df_forex.columns = df_forex.columns.get_level_values(0)
df_forex.dropna(inplace=True)

In [4]:
class Forex_Strategy(Strategy):
    # declarar valores
    sl_pct = 0.0015
    tp_pct = 0.003
    window = 500
    num_std = 3
    trend_window = 2000 # para ver tendencia macro (aprox 33 horas en 1m)
    max_duration = 240 # 4 horas para evitar quedar atrapados

    def init(self):
        # calcular indicadores a usar
        close = self.data.Close
        self.sma = self.I(lambda x: pd.Series(x).rolling(self.window).mean(), close)
        self.rolling_std = self.I(lambda x: pd.Series(x).rolling(self.window).std(), close)

        #FILTRO de tendencia macro
        self.trend_sma = self.I(lambda x: pd.Series(x).rolling(self.trend_window).mean(), close)

    def next(self):
        # estrategia
        price = self.data.Close[-1]
        media = self.sma[-1]
        std = self.rolling_std[-1]
        trend =  self.trend_sma[-1] # agregamos tendencia

        # REGLA DE SALIDA POR TIEMPO
        # si hay una posicion abierta, se revisa cuanto tiempo lleva
        if self.position:
            # calculo de cuantas velas lleva abierta la posicion
            if len(self.data) - self.trades[-1].entry_bar > self.max_duration:
               # print(f'Cierre por tiempo en la vela {len(self.data)}') # para debuggear en la consola
                self.position.close()
            return
        

        # ENTRADAS FILTRADAS POR TENDENCIA
        # solo comppramos si el precio esta encima de la tendencia macro (trend following)
        # y solo por debajo de la banda inferior (reversion de corto plazo)
        if price < (media - self.num_std * std) and (price > trend):
            self.open_long(price)

        #solo vender si el precio esta por debajo de la tendencia macro
        # y por encima de la banda superior
        elif price > (media + self.num_std * std) and (price < trend):
            self.open_short(price)

    def open_long(self, price):
        self.buy(
            sl=price * (1 - self.sl_pct),
            tp=price * (1 + self.tp_pct)
        )

    def open_short(self, price):
        self.sell(
            sl=price * (1 + self.sl_pct),
            tp=price * (1 - self.tp_pct)
        )

In [5]:
bt = Backtest(df_forex, Forex_Strategy, cash=100, commission=0.0001)
stats = bt.run()
print(stats)

/tmp/ipykernel_195246/618667144.py:1: UserWarning: Data index is not sorted in ascending order. Sorting.
  bt = Backtest(df_forex, Forex_Strategy, cash=100, commission=0.0001)


Start                     2025-01-01 17:00:00
End                       2025-12-31 16:57:00
Duration                    363 days 23:57:00
Exposure Time [%]                     3.71287
Equity Final [$]                    101.79016
Equity Peak [$]                     102.99488
Commissions [$]                       1.49439
Return [%]                            1.79016
Buy & Hold Return [%]                14.27362
Return (Ann.) [%]                     1.43878
Volatility (Ann.) [%]                  1.2536
CAGR [%]                              1.23596
Sharpe Ratio                          1.14772
Sortino Ratio                         2.06119
Calmar Ratio                          1.11365
Alpha [%]                             1.78168
Beta                                  0.00059
Max. Drawdown [%]                    -1.29195
Avg. Drawdown [%]                    -0.05217
Max. Drawdown Duration      180 days 19:00:00
Avg. Drawdown Duration        2 days 05:35:00
# Trades                          

In [6]:
optimized = bt.optimize(
    sl_pct=[0.0005, 0.001, 0.0015],
    tp_pct=[0.002, 0.003, 0.005],
    window=[100, 200, 300, 500],
    num_std=[2, 2.5, 3],
    maximize='Sharpe Ratio'
)

print(optimized)
print(optimized._strategy)

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. 

Start                     2025-01-01 17:00:00
End                       2025-12-31 16:57:00
Duration                    363 days 23:57:00
Exposure Time [%]                     3.71287
Equity Final [$]                    101.79016
Equity Peak [$]                     102.99488
Commissions [$]                       1.49439
Return [%]                            1.79016
Buy & Hold Return [%]                14.27362
Return (Ann.) [%]                     1.43878
Volatility (Ann.) [%]                  1.2536
CAGR [%]                              1.23596
Sharpe Ratio                          1.14772
Sortino Ratio                         2.06119
Calmar Ratio                          1.11365
Alpha [%]                             1.78168
Beta                                  0.00059
Max. Drawdown [%]                    -1.29195
Avg. Drawdown [%]                    -0.05217
Max. Drawdown Duration      180 days 19:00:00
Avg. Drawdown Duration        2 days 05:35:00
# Trades                          

In [7]:
bt.plot()

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '1h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1367', ...)

In [8]:
# simulacion de apalancamiento x10
bt_leverage = Backtest(df_forex, Forex_Strategy, cash=100, commission=0.0001, margin=0.1)
stats_leverage = bt_leverage.run()
print(stats_leverage)

print(f"Retorno con apalancamiento: {stats_leverage['Return [%]']: .2f}%")
print(f"Drawdown con apalancamiento: {stats_leverage['Max. Drawdown [%]']: .2f}%")

/tmp/ipykernel_195246/4274580194.py:2: UserWarning: Data index is not sorted in ascending order. Sorting.
  bt_leverage = Backtest(df_forex, Forex_Strategy, cash=100, commission=0.0001, margin=0.1)


Start                     2025-01-01 17:00:00
End                       2025-12-31 16:57:00
Duration                    363 days 23:57:00
Exposure Time [%]                     3.71287
Equity Final [$]                    118.48887
Equity Peak [$]                     133.59075
Commissions [$]                      17.32124
Return [%]                           18.48887
Buy & Hold Return [%]                14.27362
Return (Ann.) [%]                    14.63538
Volatility (Ann.) [%]                14.29191
CAGR [%]                             12.46253
Sharpe Ratio                          1.02403
Sortino Ratio                         2.08419
Calmar Ratio                          1.18587
Alpha [%]                            18.40794
Beta                                  0.00567
Max. Drawdown [%]                   -12.34143
Avg. Drawdown [%]                    -0.52078
Max. Drawdown Duration      180 days 19:00:00
Avg. Drawdown Duration        2 days 06:42:00
# Trades                          